# Exploratory Data Analysis on the Marathos Dataset

## Loading the dataset

In [0]:
# Use the raw dataset inside the volume
VOLUME_PATH = "/Volumes/marathos_catalog/default/raw/"

# Use spark and sql
spark.sql(f"LIST '{VOLUME_PATH}'")

In [0]:
# See data path
DATA_PATH = "/Volumes/marathos_catalog/default/raw/data/"

display(spark.sql(f"LIST '{DATA_PATH}'"))

In [0]:
# Show what is inside the data folder. Using this method for better debugging later.
FILE_PATH = "/Volumes/marathos_catalog/default/raw/data/TWO_CENTURIES_OF_UM_RACES.csv"

# read csv
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(FILE_PATH)
)

display(df)

## Schema Check

In [0]:
# Check the schema that Spark inferred from the CSV file | column name + data type

df.printSchema()

## Number of rows and columns

In [0]:
row_count = df.count()
column_count = len(df.columns)

print(f"Number of rows: {row_count}")
print(f"Number of columnss: {column_count}")


## Name of columns and number of distinct values

In [0]:
from pyspark.sql.functions import countDistinct

distinct_counts = []

for column in df.columns:
    distinct_count = df.agg(countDistinct(column).alias("distinct_values")).collect()[0]["distinct_values"]
    distinct_counts.append((column, distinct_count))

distinct_counts_df = spark.createDataFrame(
    distinct_counts,
    ["column_name", "distinct_values"]
)

display(distinct_counts_df)

## Descriptive Summary of numerical fields

The dataset has columns that contain numeric-looking values but still be stored as string.
I will divide these into different sections.

### Already numeric fields
```
Year of event                  integer
Event number of finishers      integer
Athlete year of birth          double
Athlete ID                     integer
```

###  Numeric but `string` type
```
Event distance/length -- investigate in EDA and clean more in silver
Athlete performance   -- investigate in EDA in Silver Layer later
Athlete average speed  -- cast to double ??
```

#### 1. Event distance/length
For basic EDA, inspect it as a categorical/text column first. Full cleaning can wait until Silver.
Need to extract things divided layter like this
```
distance_or_length_value = 50
distance_or_length_unit = km`
```

#### 2. Athlete average speed  -- cast to double


#### 3. Athelete Performance
```
4:51:39 h
5:15:45 h
```
Athlete performance contains duration values stored as strings, so I will not include it in the basic numeric summary until it is cleaned and converted. This is time duration, so it is not directly numeric yet. I will convert it later in Silver, probably into seconds, minutes, or hours.


## Descriptive summary for already numeric columns

In [0]:
# Descriptive summary of columns that Spark already inferred as numeric

already_numeric_columns = [
    "Year of event",
    "Event number of finishers",
    "Athlete year of birth"
]

display(df.select(already_numeric_columns).summary())

### Fimdings Descriptive summary of already numerical fields
I created a descriptive summary for the numerical columns that Spark inferred from the raw CSV file:
- `Year of event`
- `Event number of finishers`
- `Athlete year of birth`

- The dataset contains records from **1798 to 2022**, which aligns with the historical scope of the dataset. Most records are from more recent years, with the median event year being **2015**.

- For `Event number of finishers`, the median is **235**, while the mean is around **1452**. This shows that most events have a few hundred finishers, but some very large events increase the average. The maximum value is **20,027 finishers**.

- For `Athlete year of birth`, there are fewer non-null values compared to the other numerical columns, which means some athlete birth years are missing. The minimum value is **1193** and the maximum value is **2021**, so this column will need further validation during the Silver layer cleaning process.

- This summary helps identify the general distribution of the numerical fields and highlights potential data quality issues that should be handled later in the pipeline.

## Athlete Average Speed
Add Athlete average speed by converting it to numeric for now because Athlete average speed is currently a string, create a temporary EDA version:

In [0]:
# Check malformed average speed values

from pyspark.sql.functions import col, expr

athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

display(
    athlete_speed_df
    .filter(
        col("Athlete average speed").isNotNull()
        & col("athlete_average_speed_numeric").isNull()
    )
    .select("Athlete average speed")
    .distinct()
)

In [0]:
# Descriptive summary for Athlete average speed only
# Some values are malformed (sample above), so try_cast converts invalid values to NULL instead of failing.

from pyspark.sql.functions import expr

athlete_speed_df = df.withColumn(
    "athlete_average_speed_numeric",
    expr("try_cast(`Athlete average speed` as double)")
)

display(
    athlete_speed_df
    .select("athlete_average_speed_numeric")
    .summary()
)

### Findings descriptive summary of athlete average speed
- This column looked numerical, but Spark read it as a string. To analyze it, I temporarily converted it to a numeric column using `try_cast`.

- Most values seem reasonable. The median speed is **7.354**, and most values are between **5.783** and **9.1**.
- However, the maximum value is **29,644**, which is clearly too high for athlete average speed. This also makes the average much higher than expected.

- This tells me that the column has some incorrect or extreme values. I will handle these data quality issues later in the Silver layer.

## Event distance/length
I sse that this column is a mess. 
So the best EDA approach is:

- Count distinct values.
- Show the most common values.
- Extract the numeric part and unit temporarily for analysis.
- Count how many records are km, mi, h, etc.
- Later in Silver, create proper cleaned columns.

In [0]:
# Count distinct values in Event distance/length

distinct_distance_count = df.select("Event distance/length").distinct().count()

print(f"Number of distinct Event distance/length values: {distinct_distance_count}")

In [0]:
# Show most common Event distance/length values
# Which race types appear most often in the dataset?

from pyspark.sql.functions import col, count

event_distance_counts_df = (
    df.groupBy("Event distance/length")
    .agg(count("*").alias("row_count"))
    .orderBy(col("row_count").desc())
)

display(event_distance_counts_df)

In [0]:
# Temporarily extract numeric value and unit from Event distance/length for EDA
# Using try_cast because some values may not have a valid numeric part.

from pyspark.sql.functions import col, regexp_extract, expr

event_distance_eda_df = (
    df
    .withColumn(
        "distance_or_length_value_raw",
        regexp_extract(col("Event distance/length"), r"^([0-9]+(?:\.[0-9]+)?)", 1)
    )
    .withColumn(
        "distance_or_length_value",
        expr("try_cast(distance_or_length_value_raw as double)")
    )
    .withColumn(
        "distance_or_length_unit",
        regexp_extract(col("Event distance/length"), r"([a-zA-Z]+)$", 1)
    )
)

display(
    event_distance_eda_df
    .select(
        "Event distance/length",
        "distance_or_length_value",
        "distance_or_length_unit"
    )
    .distinct()
    .orderBy("distance_or_length_unit", "distance_or_length_value")
)